In [ ]:
import pandas as pd
import numpy as np
import os
import joblib # You will need this to save the final lists

# 1. Setup your file mapping (Ensure these paths are correct in your environment)
files = {
    "Polynomial":   "/workspaces/Laptop_Price_Predictor/notebooks/FC211007_Malanka/model_2/cleaned_data.csv",
    "LightGBM":     "/workspaces/Laptop_Price_Predictor/notebooks/FC211022_Dunith/model_5/cleaned_data_5.csv",
    "XGBoost":      "/workspaces/Laptop_Price_Predictor/notebooks/FC211037_Tharushima/cleaned_laptop_dataset_2.csv",
    "RandomForest": "/workspaces/Laptop_Price_Predictor/notebooks/FC211020_Dinusha/Final_Model_Development/cleaned_laptop_dataset_2.csv",
    "Linear":       "/workspaces/Laptop_Price_Predictor/notebooks/FC211046_Dulakshi/cleaned_data.csv"
}

def get_model_features(file_name):
    """
    Reads the CSV, intelligently encodes if necessary, and returns the final 
    list of feature names (including order).
    """
    if not os.path.exists(file_name):
        print(f"Skipping {file_name}: File not found.")
        return []
        
    print(f"Processing {file_name}...", end=" ")
    try:
        df = pd.read_csv(file_name)
        
        # REMOVE TARGET VARIABLE
        if 'Price' in df.columns:
            df = df.drop(columns=['Price'])
        if 'log_price' in df.columns:
            df = df.drop(columns=['log_price'])
            
        # --- THE SMART CHECK (If 'Brand' exists, it's RAW data) ---
        if 'Brand' in df.columns:
            print("[RAW DATA] -> Encoding...", end=" ")
            
            # 1. Manual Mapping (Ordinal/Binary)
            mappings = {
                'RAM_Expandable': {'Yes': 1, 'No': 0},
                'Display_type': {'LCD': 0, 'LED': 1},
                'Display_Tier': {'Small': 1, 'Large': 2},
                'GPU_Tier': {'Entry-level': 1, 'Low-end': 2, 'Mid-end': 3, 'High-end': 4}
            }
            
            for col, map_dict in mappings.items():
                if col in df.columns:
                    # Use .astype(str) for safety
                    df[col] = df[col].astype(str).map(map_dict)

            # Map RAM_TYPE(DDR) specifically
            if 'RAM_TYPE(DDR)' in df.columns:
                df['RAM_TYPE(DDR)'] = df['RAM_TYPE(DDR)'].astype(str).map({
                    '3': 1, 'LP3': 2, '4': 3, 'LP4': 4, '5': 5, 'LP5': 6
                })

            # 2. One-Hot Encoding
            cols_to_dummy = ['Processor_Category', 'Brand', 'Processor_Brand', 'GPU_Brand']
            existing_dummy_cols = [c for c in cols_to_dummy if c in df.columns]
            
            df = pd.get_dummies(df, columns=existing_dummy_cols, drop_first=True)
            
        else:
            print("[ALREADY ENCODED] -> Skipping encoding...", end=" ")

        # Get final list of features (ORDER MATTERS)
        features = df.columns.tolist()
        print(f"Done! ({len(features)} features)")
        return features

    except Exception as e:
        print(f"\nERROR processing {file_name}: {e}")
        return []

# 3. Execution and Comparison
feature_lists = {}
print("--- Checking Feature Consistency (Content and Order) ---\n")

for model_name, csv_file in files.items():
    feature_list = get_model_features(csv_file)
    feature_lists[model_name] = feature_list

# Determine the base model (e.g., Polynomial, provided its file was found)
base_model = "Polynomial"
while not feature_lists.get(base_model) and base_model in files:
    # Fallback to the next model if the base file was missing
    models = list(files.keys())
    base_index = models.index(base_model)
    if base_index + 1 < len(models):
        base_model = models[base_index + 1]
    else:
        print("\nCould not find any data files to use as a baseline for comparison.")
        exit()

base_list = feature_lists.get(base_model, [])
base_set = set(base_list)

print(f"\n--- Comparing all models against {base_model} ---")
all_match = True

for model_name, current_list in feature_lists.items():
    if model_name == base_model or not current_list: continue
    
    current_set = set(current_list)
    
    # 4a. Check for content mismatch (missing or extra columns)
    missing = base_set - current_set
    extra = current_set - base_set
    
    if missing or extra:
        all_match = False
        print(f"\n❌ CONTENT MISMATCH: {model_name} DOES NOT have the same features as {base_model}!")
        if missing:
            print(f"  Missing columns in {model_name}: {sorted(list(missing))}")
        if extra:
            print(f"  Extra columns in {model_name}: {sorted(list(extra))}")
        continue 

    # 4b. Check for order mismatch (if content is the same)
    if current_list != base_list:
        all_match = False
        print(f"\n⚠️ ORDER MISMATCH: {model_name} has the SAME columns as {base_model}, but in a different order!")
        for i, (a, b) in enumerate(zip(base_list, current_list)):
            if a != b:
                print(f"  Order difference starts at position {i}: Expected '{a}', Found '{b}'")
                break
    else:
        print(f"✅ {model_name} matches {base_model} (Content and Order)")

# 5. Final Conclusion and Saving
if not all_match:
    print("\nCONCLUSION: Feature content or order is NOT consistent across all models.")
    print("Action: You MUST save and load a separate feature_list.pkl for each model.")
else:
    print("\nCONCLUSION: All models share the same features and order.")
    print("Action: You can safely save one master feature list.")

# print("\n--- Saving All Feature Lists ---")
# # Always save separately to be safe, but you only need to use one if they all match.
# for model_name, feature_list in feature_lists.items():
#     if feature_list: # Only save if a list was successfully generated
#         pkl_name = f"features_{model_name}.pkl"
#         joblib.dump(feature_list, pkl_name)
#         print(f"Saved {model_name} feature list to {pkl_name}")

--- Checking Feature Consistency (Content and Order) ---

Processing /workspaces/Laptop_Price_Predictor/notebooks/FC211007_Malanka/model_2/cleaned_data.csv... [ALREADY ENCODED] -> Skipping encoding... Done! (39 features)
Processing /workspaces/Laptop_Price_Predictor/notebooks/FC211022_Dunith/model_5/cleaned_data_5.csv... [ALREADY ENCODED] -> Skipping encoding... Done! (39 features)
Processing /workspaces/Laptop_Price_Predictor/notebooks/FC211037_Tharushima/cleaned_laptop_dataset_2.csv... [RAW DATA] -> Encoding... Done! (39 features)
Processing /workspaces/Laptop_Price_Predictor/notebooks/FC211020_Dinusha/Final_Model_Development/cleaned_laptop_dataset_2.csv... [RAW DATA] -> Encoding... Done! (39 features)
Processing /workspaces/Laptop_Price_Predictor/notebooks/FC211046_Dulakshi/cleaned_data.csv... [ALREADY ENCODED] -> Skipping encoding... Done! (39 features)

--- Comparing all models against Polynomial ---
✅ LightGBM matches Polynomial (Content and Order)

⚠️ ORDER MISMATCH: XGBoost has